In [1]:
from itertools import combinations
import numpy as np

### Triangular inequality  - test




In [ ]:
import numpy as np
import glob
from itertools import combinations


print("--- TEST DI SIMMETRIA (O(N^2) - Veloce) ---")
print("Controlla se dist(A, B) == dist(B, A)")

# Prendo tutti i file
problem_files = glob.glob("LAB2-data/*.npy")
problem_files = [f for f in problem_files if 'test_problem' not in f]
problem_files.sort()

def get_name(path):
    return path.replace('\\', '/').split('/')[-1]

for problem_file_path in problem_files:
    problem_name = get_name(problem_file_path)
    problem = np.load(problem_file_path)
    
    is_symmetric = np.allclose(problem, problem.T)
    
    print(f"File: {problem_name:<20} | È Simmetrico? -> {is_symmetric}")


## HILL CLIMBING

**Obbiettivo:** Trovare il percorso con il "punteggio" (costo) più **basso** possibile.


- **g (Geometric)**: Il problema è geometrico e simmetrico. 

- **r1 e r2 (Random)**: I problemi sono asimmetrici. 

In [ ]:
import random
import time
import glob
import pandas as pd
import math
from IPython.display import display
from numba import jit




@jit(nopython=True) # <<< OTTIMIZZATO CON NUMBA
def evaluate_solution(solution, distance_matrix):
    """
    Costo totale (distanza) di un percorso (soluzione).
    """
    total_cost = 0
    num_cities = len(solution)
    for i in range(num_cities):
        city_a = solution[i]
        city_b = solution[(i + 1) % num_cities] 
        total_cost += distance_matrix[city_a, city_b]
    return total_cost



def get_random_solution(num_cities):
    """
    Genera una soluzione casuale (un percorso come lista di indici).
    """
    solution = list(range(num_cities))
    random.shuffle(solution)
    return solution



### Divisione dei problemi 

In [3]:

# Cerca tutti i file
problem_files = glob.glob("LAB2-data/*.npy")
problem_files = [f for f in problem_files if 'test_problem' not in f]

def get_name(path):
    return path.replace('\\', '/').split('/')[-1]

def get_sort_key(file_path):
    name = get_name(file_path)
    number_str = name.split('_')[-1] 
    number_str = number_str.split('.')[0]
    return int(number_str)


# 3 liste
g_files = [f for f in problem_files if get_name(f).startswith('problem_g')]
r1_files = [f for f in problem_files if get_name(f).startswith('problem_r1')]
r2_files = [f for f in problem_files if get_name(f).startswith('problem_r2')]

g_files = sorted(g_files, key=get_sort_key)
r1_files = sorted(r1_files, key=get_sort_key)
r2_files = sorted(r2_files, key=get_sort_key)

print(f"File G (Simmetrici): {len(g_files)}")
print(f"File R1 (Asimmetrici): {len(r1_files)}")
print(f"File R2 (Asimmetrici): {len(r2_files)}")



File G (Simmetrici): 7
File R1 (Asimmetrici): 7
File R2 (Asimmetrici): 7


In [ ]:
# --- Componenti dell'Algoritmo Genetico (GA) ---

def tournament_selection(population, fitnesses, k=3):
    # Seleziona k indici casuali (con rimpiazzo)
    selected_indices = [random.randrange(len(population)) for _ in range(k)]
    
    # Trova l'indice del migliore (fitness minore) tra i k selezionati
    best_index_in_tournament = min(selected_indices, key=lambda i: fitnesses[i])
    
    return population[best_index_in_tournament]



# --- Componenti dell'Algoritmo Genetico (GA) per TSP ---

''' CROSSOVER MIGLIORE ??? '''
def order_crossover(parent1, parent2):

    """ Esegue l'Order Crossover (OX1), un operatore standard per permutazioni """

    num_cities = len(parent1)
    start, end = sorted(random.sample(range(num_cities), 2))
    
    child = [None] * num_cities
    child[start:end+1] = parent1[start:end+1]
    cities_in_child = set(child)
    
    child_idx = (end + 1) % num_cities
    parent2_idx = (end + 1) % num_cities
    
    while None in child:
        city = parent2[parent2_idx]
        if city not in cities_in_child:
            child[child_idx] = city
            child_idx = (child_idx + 1) % num_cities
        parent2_idx = (parent2_idx + 1) % num_cities
        
    return child




''' MUTAZIONE MIGLIORE??? '''
## swap/insert mutation
def mutation_hybrid(solution):
    """
    Applica una mutazione scelta a caso tra 'swap' e 'insert'.
    """
    if random.random() < 0.8:
        # --- Swap Mutation ---
        i, k = random.sample(range(len(solution)), 2)
        solution[i], solution[k] = solution[k], solution[i]
    else:
        # --- Insert Mutation ---
        i, k = random.sample(range(len(solution)), 2)
        
        city_to_move = solution.pop(i)
        solution.insert(k, city_to_move)
        
    return solution

In [ ]:
@jit(nopython=True) #  OTTIMIZZATO CON NUMBA
def apply_local_search_2opt_fast(solution, distance_matrix):
    """
    Applica una ricerca locale 2-opt (Hill Climbing).
    Itera finché non trova un miglioramento.
    """
    num_cities = len(solution)
    current_cost = evaluate_solution(solution, distance_matrix)
    
    improved = True
    while improved:
        improved = False
        
        for i in range(num_cities - 1):
            for k in range(i + 1, num_cities):
                
                # Evita il caso che rompe lo stesso arco (wrap-around)
                if i == 0 and k == num_cities - 1:
                    continue

                city_i_minus_1 = solution[i - 1]
                city_i = solution[i]
                city_k = solution[k]
                city_k_plus_1 = solution[(k + 1) % num_cities]

                cost_removed = distance_matrix[city_i_minus_1, city_i] + distance_matrix[city_k, city_k_plus_1]
                cost_added = distance_matrix[city_i_minus_1, city_k] + distance_matrix[city_i, city_k_plus_1]
                
                delta_cost = cost_added - cost_removed
                # -----------------------------------------------------------

                if delta_cost < -1e-9: # Se c'è un miglioramento 
                    # mossa 2-opt (inversione)
                    solution = solution[:i] + solution[i:k+1][::-1] + solution[k+1:]
                    current_cost += delta_cost
                    improved = True
                    break 
            if improved:
                break 
                
    return solution, current_cost



# --- O(1) per ATSP) ---
@jit(nopython=True) # <<< OTTIMIZZATO CON NUMBA
def apply_local_search_insert_ATSP(solution, distance_matrix):
    """
    Applica una ricerca locale (Hill Climbing) usando la mossa 'insert'.
    Itera finché non trova un miglioramento.
    """
    num_cities = len(solution)
    current_cost = evaluate_solution(solution, distance_matrix) # Calcolo iniziale
    
    improved = True
    while improved:
        improved = False
        
        for k in range(num_cities): 
            
            city_c = solution[k]
            city_y = solution[k - 1] 
            city_z = solution[(k + 1) % num_cities]

            for i in range(num_cities):
                
                if i == k or i == (k - 1 + num_cities) % num_cities:
                    continue
                
                city_a = solution[i]
                city_b = solution[(i + 1) % num_cities]

                cost_removed = distance_matrix[city_y, city_c] + distance_matrix[city_c, city_z] + distance_matrix[city_a, city_b]
                cost_added   = distance_matrix[city_y, city_z] + distance_matrix[city_a, city_c] + distance_matrix[city_c, city_b]
                
                delta_cost = cost_added - cost_removed

                if delta_cost < -1e-9: 
                    
                    sol_list = list(solution) 
                    
                    city_to_move = sol_list.pop(k)
                    
                    new_i = sol_list.index(city_a)
                    
                    sol_list.insert(new_i + 1, city_to_move)

                    solution = sol_list 
                    current_cost += delta_cost
                    improved = True
                    break 
            if improved:
                break 
                
    return solution, current_cost

In [ ]:
# --- ALGORITHM (GA + Local Search) ---

def memetic_algorithm_2opt(distance_matrix, 
                           population_size=100, 
                           generations=500, 
                           elite_size=10, 
                           mutation_rate=0.1,
                           local_search_rate=0.2):
    """
    Algoritmo (GA + 2-opt Local Search).
    - elitismo: i 'elite_size' migliori sopravvivono sempre.
    """

    num_cities = distance_matrix.shape[0]
    
    # --- 1. Inizializzazione ---
    population = []
    fitnesses = []
    for _ in range(population_size):
        solution = get_random_solution(num_cities)
        fitness = evaluate_solution(solution, distance_matrix)
        population.append(solution)
        fitnesses.append(fitness)
        
    # Teniamo traccia del migliore di sempre
    best_solution = min(population, key=lambda sol: evaluate_solution(sol, distance_matrix))
    best_cost = evaluate_solution(best_solution, distance_matrix)
    

    # --- 2. Ciclo Evolutivo ---
    for gen in range(generations):
        new_population = []
        new_fitnesses = []
        
        # --- 3. Elitismo ---
        sorted_indices = sorted(range(population_size), key=lambda k: fitnesses[k])
        
        for i in range(elite_size):
            elite_index = sorted_indices[i]
            new_population.append(population[elite_index])
            new_fitnesses.append(fitnesses[elite_index])
            
        # --- 4. Generazione Nuovi Figli ---
        while len(new_population) < population_size:
            # Selezione
            parent1 = tournament_selection(population, fitnesses)
            parent2 = tournament_selection(population, fitnesses)
            
            # Crossover
            child = order_crossover(parent1, parent2)
            
            # Mutazione
            if random.random() < mutation_rate:
                child = mutation_hybrid(child)
            
            # --- 5. Local Search (Parte Memetica) ---
            if random.random() < local_search_rate:
                child, child_cost = apply_local_search_2opt_fast(child, distance_matrix)
            else:
                child_cost = evaluate_solution(child, distance_matrix)

            new_population.append(child)
            new_fitnesses.append(child_cost)

        # Aggiorna la popolazione
        population = new_population
        fitnesses = new_fitnesses
        
        # Aggiorna il migliore di sempre
        current_best_index = min(range(population_size), key=lambda k: fitnesses[k])
        current_best_cost = fitnesses[current_best_index]
        
        if current_best_cost < best_cost:
            best_cost = current_best_cost
            best_solution = population[current_best_index]
            
    return best_solution, best_cost




# --- Algoritmo 5: ALGORITHM per ATSP (r1_, r2_) ---

 
def memetic_algorithm_ATSP(distance_matrix, 
                           population_size=100, 
                           generations=500, 
                           elite_rate=0.2, 
                           mutation_rate=0.1,
                           local_search_rate=0.2,
                           tournament_k=5):
    """
    Algoritmo Memetico (GA + 'insert' Local Search).
    *** SPECIFICO PER ATSP (r1_, r2_) ***
    """
    num_cities = distance_matrix.shape[0]
    
    # --- 1. Inizializzazione ---
    population = []
    for _ in range(population_size):
        population.append(get_random_solution(num_cities))
        
    best_solution_ever = population[0]
    best_cost_ever = evaluate_solution(best_solution_ever, distance_matrix)

    # --- 2. Ciclo Evolutivo ---
    for gen in range(generations):
        
        # --- 3. Valutazione ---
        fitnesses = [evaluate_solution(sol, distance_matrix) for sol in population]
        
        # --- 4. Ordinamento e Logica Elitismo ---
        sorted_indices = np.argsort(fitnesses)
        population = [population[i] for i in sorted_indices]
        fitnesses = [fitnesses[i] for i in sorted_indices]

        if fitnesses[0] < best_cost_ever:
            best_cost_ever = fitnesses[0]
            best_solution_ever = population[0]

        elite_size = int(population_size * elite_rate)
        new_population = [population[i].copy() for i in range(elite_size)]
        
        elite_pool = population[:elite_size]
        elite_fitnesses = fitnesses[:elite_size]

        # --- 5. Generazione Nuovi Figli ---
        while len(new_population) < population_size:
            
            # Selezione
            parent1 = tournament_selection(elite_pool, elite_fitnesses, k=tournament_k)
            parent2 = tournament_selection(elite_pool, elite_fitnesses, k=tournament_k)
            
            # Crossover 
            child = order_crossover(parent1, parent2)
            
            # Mutazione (specifica per TSP)
            if random.random() < mutation_rate:
                child = mutation_hybrid(child) 
            
            # --- 6. Local Search ---
            if random.random() < local_search_rate:
                # Applichiamo la NUOVA local search
                child, _ = apply_local_search_insert_ATSP(child, distance_matrix)

            new_population.append(child)

        population = new_population
            
    # Alla fine, rivaluta il migliore
    final_fitnesses = [evaluate_solution(sol, distance_matrix) for sol in population]
    best_index = np.argmin(final_fitnesses)
    final_best_solution = population[best_index]
    final_best_cost = final_fitnesses[best_index]

    if best_cost_ever < final_best_cost:
        return best_solution_ever, best_cost_ever
    else:
        return final_best_solution, final_best_cost

In [8]:

print("\n--- ESECUZIONE PROBLEMI GEOMETRICI (g) ---")
g_results = []

for problem_file_path in g_files:
    problem_name = get_name(problem_file_path)
    problem_matrix = np.load(problem_file_path)
    
    print(f"\n Processando (MA-2opt-FAST): {problem_name}")
    start_time = time.time()
    
    ma_sol, ma_cost = memetic_algorithm_2opt(
        problem_matrix,
        population_size=200,  # inizio 100 
        generations=400, # Aumenta per problemi più grandi
        elite_size=15,   ## su population 200
        mutation_rate=0.1,
        local_search_rate=0.25 # Applica local search al 25% dei figli
    )
    
    ma_time = time.time() - start_time
    
    print(f"    -> Risultato: Costo={ma_cost:.2f}, Tempo={ma_time:.4f}s")
    
    g_results.append({
        'Problem': problem_name,
        'MA Cost': ma_cost,
        'MA Time (s)': ma_time
    })

print("\n--- Risultati Categoria G (Memetico) ---")
g_df = pd.DataFrame(g_results)
display(g_df)



--- ESECUZIONE PROBLEMI GEOMETRICI (g_) - ALGORITMO MEMETICO ---

 Processando (MA-2opt-FAST): problem_g_10.npy
    -> Risultato: Costo=1497.66, Tempo=0.7460s

 Processando (MA-2opt-FAST): problem_g_20.npy
    -> Risultato: Costo=1755.51, Tempo=1.2268s

 Processando (MA-2opt-FAST): problem_g_50.npy
    -> Risultato: Costo=2629.99, Tempo=2.3728s

 Processando (MA-2opt-FAST): problem_g_100.npy
    -> Risultato: Costo=3957.49, Tempo=5.6652s

 Processando (MA-2opt-FAST): problem_g_200.npy
    -> Risultato: Costo=5422.41, Tempo=12.9879s

 Processando (MA-2opt-FAST): problem_g_500.npy
    -> Risultato: Costo=8320.24, Tempo=102.7233s

 Processando (MA-2opt-FAST): problem_g_1000.npy
    -> Risultato: Costo=11768.39, Tempo=674.8836s

--- Risultati Categoria G (Memetico) ---


,Problem,MA Cost,MA Time (s)
0,problem_g_10.npy,1497.663648,0.746032
1,problem_g_20.npy,1755.514677,1.226844
2,problem_g_50.npy,2629.986686,2.372823
3,problem_g_100.npy,3957.492539,5.665192
4,problem_g_200.npy,5422.414308,12.987909
5,problem_g_500.npy,8320.242597,102.723349
6,problem_g_1000.npy,11768.389705,674.883598


In [ ]:
'''
        population_size=200,  
        generations=400,
        elite_size=15, ## su population 200
        mutation_rate=0.1,
        local_search_rate=0.25 
'''



# --- ESECUZIONE PROBLEMI ASIMMETRICI (r1)  ---

print("\n--- ESECUZIONE PROBLEMI ASIMMETRICI (r1) ---")
r1_results = []

for problem_file_path in r1_files:
    problem_name = get_name(problem_file_path)
    problem_matrix = np.load(problem_file_path)
    
    print(f"\n Processando (MA-Insert-FAST): {problem_name}")
    start_time = time.time()
    
    # REGOLA I PARAMETRI QUI
    ma_sol, ma_cost = memetic_algorithm_ATSP(
        problem_matrix,
        population_size=200,  
        generations=450,    # Aumenta per problemi più grandi
        elite_rate=0.11,     # 20% di élite
        mutation_rate=0.2,   # 10% 
        local_search_rate=0.20, # Applica 'insert' al 25% dei figli
        tournament_k=5
    )
    
    ma_time = time.time() - start_time
    
    print(f"    -> Risultato: Costo={ma_cost:.2f}, Tempo={ma_time:.4f}s")
    
    r1_results.append({
        'Problem': problem_name,
        'MA Cost': ma_cost,
        'MA Time (s)': ma_time
    })

print("\n--- Risultati Categoria R1 (Memetico) ---")
r1_df = pd.DataFrame(r1_results)
display(r1_df)


--- ESECUZIONE PROBLEMI ASIMMETRICI (r1_) - MA (Asimmetrico) ---

 Processando (MA-Insert-FAST): problem_r1_10.npy
    -> Risultato: Costo=184.27, Tempo=1.2266s

 Processando (MA-Insert-FAST): problem_r1_20.npy
    -> Risultato: Costo=337.29, Tempo=1.4290s

 Processando (MA-Insert-FAST): problem_r1_50.npy
    -> Risultato: Costo=558.04, Tempo=3.0193s

 Processando (MA-Insert-FAST): problem_r1_100.npy
    -> Risultato: Costo=702.80, Tempo=6.1163s

 Processando (MA-Insert-FAST): problem_r1_200.npy
    -> Risultato: Costo=1012.73, Tempo=13.9884s

 Processando (MA-Insert-FAST): problem_r1_500.npy
    -> Risultato: Costo=1809.14, Tempo=60.8159s

 Processando (MA-Insert-FAST): problem_r1_1000.npy
    -> Risultato: Costo=3337.51, Tempo=306.5847s

--- Risultati Categoria R1 (Memetico) ---


,Problem,MA Cost,MA Time (s)
0,problem_r1_10.npy,184.273441,1.226613
1,problem_r1_20.npy,337.294872,1.429031
2,problem_r1_50.npy,558.039596,3.019262
3,problem_r1_100.npy,702.797189,6.116293
4,problem_r1_200.npy,1012.731107,13.988372
5,problem_r1_500.npy,1809.136027,60.815880
6,problem_r1_1000.npy,3337.506671,306.584696


In [ ]:
'''
R1
        population_size=200,  
        generations=450,    
        elite_rate=0.11,     
        mutation_rate=0.2,   
        local_search_rate=0.20, 
        tournament_k=5
'''

In [ ]:
print("\n--- ESECUZIONE PROBLEMI ASIMMETRICI (r2) ---")
r2_results = []

for problem_file_path in r2_files:
    problem_name = get_name(problem_file_path)
    problem_matrix = np.load(problem_file_path)
    
    print(f"\n Processando (MA-Insert-FAST-JIT): {problem_name}")
    start_time = time.time()
    
    ma_sol, ma_cost = memetic_algorithm_ATSP(
        problem_matrix,
        population_size=200,  
        generations=450,    # Aumenta per problemi più grandi
        elite_rate=0.15,     # 20% di élite
        mutation_rate=0.2,   # 10% 
        local_search_rate=0.20, # Applica 'insert' al 25% dei figli
        tournament_k=5
    )
    
    ma_time = time.time() - start_time
    
    print(f"    -> Risultato: Costo={ma_cost:.2f}, Tempo={ma_time:.4f}s")
    
    r2_results.append({
        'Problem': problem_name,
        'MA Cost': ma_cost,
        'MA Time (s)': ma_time
    })

print("\n--- Risultati Categoria R2 (Memetico) ---")
r2_df = pd.DataFrame(r2_results)
display(r2_df)


--- ESECUZIONE PROBLEMI ASIMMETRICI (r2_) - MA (Asimmetrico) ---

 Processando (MA-Insert-FAST-JIT): problem_r2_10.npy
    -> Risultato: Costo=-411.70, Tempo=0.9133s

 Processando (MA-Insert-FAST-JIT): problem_r2_20.npy
    -> Risultato: Costo=-861.67, Tempo=1.3831s

 Processando (MA-Insert-FAST-JIT): problem_r2_50.npy
    -> Risultato: Costo=-2264.85, Tempo=2.9352s

 Processando (MA-Insert-FAST-JIT): problem_r2_100.npy
    -> Risultato: Costo=-4695.53, Tempo=6.0711s

 Processando (MA-Insert-FAST-JIT): problem_r2_200.npy
    -> Risultato: Costo=-9457.48, Tempo=13.3804s

 Processando (MA-Insert-FAST-JIT): problem_r2_500.npy
    -> Risultato: Costo=-23769.84, Tempo=50.2230s

 Processando (MA-Insert-FAST-JIT): problem_r2_1000.npy
    -> Risultato: Costo=-47807.42, Tempo=205.5545s

--- Risultati Categoria R2 (Memetico) ---


,Problem,MA Cost,MA Time (s)
0,problem_r2_10.npy,-411.701716,0.913319
1,problem_r2_20.npy,-861.667206,1.383095
2,problem_r2_50.npy,-2264.853056,2.935205
3,problem_r2_100.npy,-4695.534714,6.071149
4,problem_r2_200.npy,-9457.479187,13.380420
5,problem_r2_500.npy,-23769.842101,50.223049
6,problem_r2_1000.npy,-47807.420631,205.554488


In [ ]:
''' 
COSI è PEGGIORATO ---  rimetti elite_rate 0.15 )
R2
        population_size=200,  
        generations=450,    
        elite_rate=0.15,      #MEGLIO rispetto a 0.111
        mutation_rate=0.2,   
        local_search_rate=0.20, 
        tournament_k=5
'''